# House Price Prediction

This notebook builds the machine learning part of the House Price Prediction project.

The main steps are:
- load and inspect the dataset
- clean the price and property-area fields
- do some exploratory analysis
- create useful features
- train and compare two regression models
- evaluate them on a test set
- export the best pipeline for the FastAPI backend


> Reproducibility: run the notebook top-to-bottom. Put `house_prices.csv` in `notebooks/data/` before running, or place it in the notebook working directory.


## 1. Imports and setup

In [ ]:
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 2. Load the dataset

If the CSV is already in the Colab working directory, the first line is enough.
The upload option is useful when running the notebook in a new Colab session.


In [ ]:
from pathlib import Path

# The project guide recommends notebooks/data/house_prices.csv.
# The fallback paths make the notebook easier to run from VS Code/Jupyter.
candidates = [
    Path("data/house_prices.csv"),
    Path("notebooks/data/house_prices.csv"),
    Path("house_prices.csv"),
]

csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "house_prices.csv was not found. Download the dataset and place it in "
        "notebooks/data/house_prices.csv."
    )

df = pd.read_csv(csv_path)

print("Dataset path:", csv_path)
print("Shape:", df.shape)
df.head()


## 3. Initial inspection

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst few rows:")
display(df.head())

print("\nMissing values (%):")
missing = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing.to_frame("missing_percent"))

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 4. Cleaning helper functions

The dataset stores the target price and several area/floor values as text. I convert them before training instead of trying to model the raw strings.


In [ ]:
def parse_amount(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower().replace(",", "")
    if text in {"", "nan", "none", "call for price", "call for price."}:
        return np.nan

    try:
        if "cr" in text:
            number = float(text.replace("cr", "").strip())
            return number * 10_000_000

        if "lac" in text or "lakh" in text:
            text = text.replace("lac", "").replace("lakh", "").strip()
            return float(text) * 100_000

        return float(text)
    except ValueError:
        return np.nan


def parse_area_sqft(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower().replace(",", "")
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", text)

    if not match:
        return np.nan

    number = float(match.group(1))

    if "sqm" in text or "sq. m" in text or "m2" in text:
        return number * 10.764

    return number


def parse_floor(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if "ground" in text:
        return 0

    match = re.search(r"(\d+)", text)
    if match:
        return int(match.group(1))

    return np.nan


def parse_numeric(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", str(value))
    return float(match.group(1)) if match else np.nan


## 5. Clean the main fields

In [ ]:
# Target
df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)

# Area and floor
df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area_sqft)
df["floor_num"] = df["Floor"].apply(parse_floor)

# Numeric fields
df["bathroom"] = df["Bathroom"].apply(parse_numeric)
df["balcony"] = df["Balcony"].apply(parse_numeric)

# Car parking is parsed as well. It is not used as a model feature below,
# because the project guide's requested model inputs do not include it.
df["car_parking_num"] = df["Car Parking"].apply(parse_numeric)

# Remove rows where the target cannot be used.
df = df.dropna(subset=["price_clean"]).copy()

print("Rows after removing unusable target prices:", len(df))
display(df[[
    "Amount(in rupees)", "price_clean",
    "Carpet Area", "carpet_area_sqft",
    "Floor", "floor_num",
    "Bathroom", "bathroom",
    "Balcony", "balcony",
    "Car Parking", "car_parking_num"
]].head(10))


## 6. Exploratory Data Analysis

The target is strongly right-skewed, so the price distribution is shown on a logarithmic x-axis.


**Interpretation:** The distribution is strongly right-skewed, with many listings at lower price levels and fewer very expensive properties. The log scale makes the long tail easier to inspect and motivates the later use of robust cleaning and test-set evaluation.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["price_clean"], bins=60, log_scale=True)
plt.title("House price distribution")
plt.xlabel("Price (rupees, log scale)")
plt.ylabel("Number of listings")
plt.show()


### Price vs. carpet area

**Interpretation:** The scatter plot shows the overall relationship between carpet area and price. Prices generally increase as area increases, but the spread becomes wider for larger properties, indicating that location and other categorical features also influence price.


In [ ]:
eda_area = df[
    (df["carpet_area_sqft"] > 0) &
    (df["carpet_area_sqft"] < df["carpet_area_sqft"].quantile(0.99))
].copy()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=eda_area.sample(min(5000, len(eda_area)), random_state=RANDOM_STATE),
    x="carpet_area_sqft",
    y="price_clean",
    alpha=0.35
)
plt.title("Price vs. carpet area")
plt.xlabel("Carpet area (sqft)")
plt.ylabel("Price (rupees)")
plt.show()


### Average price by the 15 most common locations

**Interpretation:** This comparison shows that average prices vary substantially across locations. This supports keeping location as a model feature and grouping less frequent locations into `Other` to reduce high-cardinality noise.


In [ ]:
top_locations = df["location"].value_counts().head(15).index
location_avg = (
    df[df["location"].isin(top_locations)]
    .groupby("location")["price_clean"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10, 6))
location_avg.plot(kind="barh")
plt.title("Average price for the top 15 locations")
plt.xlabel("Average price (rupees)")
plt.ylabel("Location")
plt.show()


### Price by furnishing status

**Interpretation:** The box plot compares prices across furnishing categories. The distributions overlap, but differences in their centers and spreads suggest furnishing status contains useful predictive information.


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x="Furnishing",
    y="price_clean",
    showfliers=False
)
plt.yscale("log")
plt.title("Price by furnishing status")
plt.xlabel("Furnishing")
plt.ylabel("Price (log scale)")
plt.xticks(rotation=15)
plt.show()


### Price by number of bathrooms

**Interpretation:** Properties with more bathrooms tend to have higher prices, although there is still substantial variation within each bathroom count. This indicates bathroom count is useful but not sufficient on its own to explain price.


In [ ]:
bath_plot = df[df["bathroom"].notna() & (df["bathroom"] <= 6)].copy()

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=bath_plot,
    x="bathroom",
    y="price_clean",
    showfliers=False
)
plt.yscale("log")
plt.title("Price by number of bathrooms")
plt.xlabel("Bathrooms")
plt.ylabel("Price (log scale)")
plt.show()


## 7. Feature engineering and outlier removal

Locations have many different values, so I keep the 50 most frequent locations and group the rest as `Other`.

For the outlier check, I use price per square foot and remove values below the 1st percentile or above the 99th percentile, as suggested in the project guide.


In [ ]:
# Keep only sensible positive values for the area calculation.
df = df[
    (df["carpet_area_sqft"].notna()) &
    (df["carpet_area_sqft"] > 0)
].copy()

df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]

low_ppsf = df["price_per_sqft"].quantile(0.01)
high_ppsf = df["price_per_sqft"].quantile(0.99)

before = len(df)
df = df[
    df["price_per_sqft"].between(low_ppsf, high_ppsf)
].copy()

print("Rows before outlier removal:", before)
print("Rows after outlier removal:", len(df))
print("Price per sqft limits:", low_ppsf, "to", high_ppsf)

# Location grouping
top_50_locations = df["location"].value_counts().head(50).index
df["location_grouped"] = df["location"].where(
    df["location"].isin(top_50_locations),
    "Other"
)

print("\nNumber of grouped locations:", df["location_grouped"].nunique())
display(df["location_grouped"].value_counts().head(10))


## 8. Prepare the modeling data

I use the same core inputs specified in the project guide. The original text columns such as title and description are not useful as direct inputs for this version of the model.


In [ ]:
numeric_features = [
    "carpet_area_sqft",
    "floor_num",
    "bathroom",
    "balcony"
]

categorical_features = [
    "location_grouped",
    "Furnishing",
    "Transaction",
    "Ownership",
    "facing"
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["price_clean"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 9. Preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)


## 10. Train two models

Linear Regression is used as a simple baseline. Random Forest is then used as a nonlinear model that can capture relationships that a straight-line model may miss.


In [ ]:
linear_model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

forest_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

print("Training Linear Regression...")
linear_model.fit(X_train, y_train)

print("Training Random Forest...")
forest_model.fit(X_train, y_train)

print("Training finished.")


## 11. Evaluate the models

In [ ]:
def evaluate_model(name, model):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }, predictions


linear_result, linear_pred = evaluate_model(
    "Linear Regression",
    linear_model
)

forest_result, forest_pred = evaluate_model(
    "Random Forest",
    forest_model
)

results = pd.DataFrame([linear_result, forest_result])
display(results)


## 12. Model comparison

For MAE and RMSE, lower values are better. For R², a higher value is better.


In [ ]:
results.sort_values("RMSE")


In [ ]:
best_model_name = results.sort_values("RMSE").iloc[0]["Model"]

if best_model_name == "Random Forest":
    best_model = forest_model
    best_pred = forest_pred
else:
    best_model = linear_model
    best_pred = linear_pred

print("Selected model:", best_model_name)


## 13. Predicted vs. actual prices

**Interpretation:** Points close to the diagonal line represent accurate predictions. The amount of scatter around the line shows the remaining prediction error; large deviations indicate properties whose prices are harder for the selected model to estimate.


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, best_pred, alpha=0.25)

lower = min(y_test.min(), best_pred.min())
upper = max(y_test.max(), best_pred.max())

plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title(f"Predicted vs. actual prices - {best_model_name}")
plt.show()


## 14. Final test metrics

In [ ]:
final_mae = mean_absolute_error(y_test, best_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
final_r2 = r2_score(y_test, best_pred)

print(f"Model: {best_model_name}")
print(f"MAE : {final_mae:,.2f}")
print(f"RMSE: {final_rmse:,.2f}")
print(f"R²  : {final_r2:.4f}")


## 15. Export the model and allowed locations

The exported object is the complete scikit-learn pipeline, including the preprocessing steps. This means the backend can pass a normal DataFrame to the model without recreating the one-hot encoding manually.


In [ ]:
from pathlib import Path
import json

# Save artifacts directly into the backend model directory so the API can use
# the exact pipeline and location list produced by this notebook.
cwd = Path.cwd()
if (cwd / "backend").is_dir():
    model_dir = cwd / "backend" / "models"
elif (cwd.parent / "backend").is_dir():
    model_dir = cwd.parent / "backend" / "models"
else:
    raise FileNotFoundError(
        "Could not locate the project backend/models directory. "
        "Run this notebook from the project root or notebooks directory."
    )

model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, model_dir / "house_price.pkl")

allowed_locations = sorted(df["location_grouped"].unique().tolist())
with open(model_dir / "locations.json", "w", encoding="utf-8") as f:
    json.dump(allowed_locations, f, indent=2)

print("Saved:")
print(model_dir / "house_price.pkl")
print(model_dir / "locations.json")


## 16. Reload check

This is a small sanity check to make sure the saved pipeline can be loaded and used for prediction.


In [ ]:
loaded_model = joblib.load(model_dir / "house_price.pkl")

sample = X_test.iloc[[0]]
sample_prediction = loaded_model.predict(sample)[0]

print("Reloaded prediction:", sample_prediction)
print("Actual value:", y_test.iloc[0])


## Conclusion

The final model is selected using the test-set comparison above. In the final report, the better model should be justified using MAE, RMSE, and R² rather than choosing a model only because it is more complex.

The exported `house_price.pkl` is the model artifact used by the FastAPI backend.
